## Step 1: Read Silver Table

In [0]:
from pyspark.sql.functions import *

df_silver = spark.table("nyc_taxi.silver.taxi_trip_silver")

print(f"Count is {df_silver.count()}")


Count is 7668647


## Step 2: Create dim_date

In [0]:
dim_date = (
    df_silver
    .select("pickup_date")
    .distinct()
    .withColumn("date_key",date_format("pickup_date","yyyyMMdd").cast("int"))
    .withColumn("day",dayofmonth("pickup_date"))
    .withColumn("month",month("pickup_date"))
    .withColumn("year",year("pickup_date"))
    .withColumn("quarter",quarter("pickup_date"))
)

In [0]:
dim_date.write.mode("overwrite").format("delta").saveAsTable("nyc_taxi.gold.dim_date")

In [0]:
%sql
select * from nyc_taxi.gold.dim_date limit 10

pickup_date,date_key,day,month,year,quarter
2026-02-02,20260202,2,2,2026,1
2025-12-31,20251231,31,12,2025,4
2026-02-24,20260224,24,2,2026,1
2026-01-04,20260104,4,1,2026,1
2026-01-21,20260121,21,1,2026,1
2026-02-25,20260225,25,2,2026,1
2026-03-10,20260310,10,3,2026,1
2026-01-20,20260120,20,1,2026,1
2026-03-05,20260305,5,3,2026,1
2026-01-13,20260113,13,1,2026,1


## Step 3: Create dim_hour

In [0]:
dim_hour  = (
    df_silver.select("pickup_hour")
            .distinct()
            .withColumnRenamed("pickup_hour","hour_key")
)

In [0]:
dim_hour = dim_hour.withColumn("time_band",
                               when(col("hour_key").between(0,5),"Night")
                               .when(col("hour_key").between(6,11),"Morning")
                               .when(col("hour_key").between(12,17),"Afternoon")
                               .otherwise("Evening"))


In [0]:
dim_hour.write.mode("overwrite").format("delta").saveAsTable("nyc_taxi.gold.dim_hour")

## Step 4: Create dim_trip_category

In [0]:
dim_trip_category = (
    df_silver.select("trip_category").distinct()
)

In [0]:
#Adding surrogate key

from pyspark.sql.window import Window
from pyspark.sql.functions import *

dim_trip_category = (
    dim_trip_category
    .withColumn("trip_category_key", row_number().over(Window.orderBy("trip_category")))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dim_trip_category.write.mode("overwrite").format("delta").saveAsTable("nyc_taxi.gold.dim_trip_category")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Step 5: Create dim_payment_type

In [0]:
dim_payment_type = df_silver.select("payment_type").distinct()

In [0]:
(dim_payment_type.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.dim_payment_type"))

## Step 6: Create Fact Table

In [0]:
fact_trip = (
    df_silver.withColumn("date_key",
                         date_format("pickup_date","yyyyMMdd").cast("int"))
)

fact_trip = fact_trip.select("date_key","pickup_hour","trip_category","payment_type","trip_distance",
                             "trip_duration_minutes","fare_amount","tip_amount","total_amount")

In [0]:
(
    fact_trip.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "nyc_taxi.gold.fact_trip"
    )
)

In [0]:
%sql
select count(*) from nyc_taxi.gold.fact_trip

count(*)
7668647


In [0]:
%sql
select * from nyc_taxi.gold.fact_trip limit 10;

date_key,pickup_hour,trip_category,payment_type,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount
20260301,0,Short Trip,1,0.76,8.55,9.3,3.01,18.06
20260301,0,Short Trip,1,1.72,9.666666666666666,12.1,3.42,20.52
20260301,0,Short Trip,1,0.43,3.8333333333333335,5.8,1.0,12.55
20260301,0,Short Trip,1,1.61,9.05,11.4,3.43,20.58
20260301,0,Medium Trip,1,3.33,16.566666666666666,18.4,4.83,28.98
20260301,0,Short Trip,1,1.22,7.466666666666667,9.3,3.01,18.06
20260301,0,Short Trip,2,0.36,2.7333333333333334,4.4,0.0,10.15
20260301,0,Short Trip,1,1.76,12.866666666666667,12.8,2.4,20.95
20260301,0,Medium Trip,2,6.73,14.733333333333333,28.2,0.0,33.95
20260301,0,Short Trip,1,1.21,11.633333333333333,11.4,5.14,22.29


In [0]:
monthly_revenue_summary = (
    spark.table("nyc_taxi.gold.fact_trip")
    .groupBy("date_key")
    .agg(
        sum("total_amount").alias("total_revenue"),
        count("*").alias("total_trips"),
        avg("fare_amount").alias("avg_fare")
    )
)

In [0]:
monthly_revenue_summary.write.mode("overwrite").format("delta").saveAsTable("nyc_taxi.gold.monthly_revenue_summary")